# XIX-Upscaler — Video (Colab Experimental)

Fitur eksperimental ini memakai akun, Google Drive, GPU, dan kuota Colab milik pengguna sendiri. Pilih **Runtime → Run all**. Notebook akan meminta izin Drive, memeriksa GPU, memasang worker XIX yang sudah diverifikasi, lalu memproses antrean video. Tidak ada pengaturan yang perlu diubah di notebook.

> Gunakan runtime GPU. Biarkan tab Colab terbuka sampai pekerjaan selesai. Jika sesi terputus, jalankan **Run all** lagi untuk melanjutkan dari checkpoint.

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys
import urllib.request
import torch

assert torch.cuda.is_available(), "GPU tidak tersedia. Pilih Runtime → Change runtime type → GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")

WORKER_VERSION = "0.1.0"
RELEASE_COMMIT = "fdc5ee84a790ba47f575fb58fa313aed6e86ce83"
WHEEL_NAME = "xix_colab_worker-0.1.0-py3-none-any.whl"
WHEEL_URL = f"https://raw.githubusercontent.com/mfahryf/xix-upscaler-colab/{RELEASE_COMMIT}/dist/{WHEEL_NAME}"
WHEEL_SIZE = 30259
WHEEL_SHA256 = "82f42421fb1bd805c3fba0f43bf71b75498495c27668844491c115e057fd08b5"
WHEEL_PATH = Path("/content") / WHEEL_NAME
JOBS_DIR = Path("/content/drive/MyDrive/XIX-Upscaler/jobs")
CACHE_DIR = Path("/content/xix-upscaler-cache")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
JOBS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Folder antrean siap: {JOBS_DIR}")

In [ ]:
with urllib.request.urlopen(WHEEL_URL, timeout=60) as response:
    wheel_bytes = response.read()
if len(wheel_bytes) != WHEEL_SIZE:
    raise RuntimeError("Ukuran paket worker tidak cocok")
if hashlib.sha256(wheel_bytes).hexdigest() != WHEEL_SHA256:
    raise RuntimeError("SHA-256 paket worker tidak cocok")
WHEEL_PATH.write_bytes(wheel_bytes)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(WHEEL_PATH)],
    check=True,
)
print(f"Worker XIX {WORKER_VERSION} terpasang dan terverifikasi")

In [ ]:
subprocess.run(
    [sys.executable, "-m", "xix_colab_worker", "verify-environment", "--jobs-dir", str(JOBS_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "xix_colab_worker", "verify-locks", "--cache-dir", str(CACHE_DIR)],
    check=True,
)

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "xix_colab_worker",
        "run-queue",
        "--jobs-dir",
        str(JOBS_DIR),
        "--cache-dir",
        str(CACHE_DIR),
    ],
    check=False,
)
if result.returncode not in (0, 1):
    raise RuntimeError(f"Worker berhenti dengan kode {result.returncode}")
print("Antrean selesai. Hasil tersedia di folder job Google Drive.")